In [1]:
!pip install staticmap pillow

In [2]:
import os
import glob
import shutil
import requests
import pandas as pd
import sys
import subprocess
from google.colab import files
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import tensorflow as tf
from staticmap import StaticMap, CircleMarker

# Carga de archivo Excel con la información

In [3]:
print("Por favor, selecciona y sube tu archivo Excel:")
subido = files.upload()

Por favor, selecciona y sube tu archivo Excel:


Saving Asaltos 1 (2).xlsx to Asaltos 1 (2).xlsx


# Obtener el archivo cargado

In [4]:
nombre_archivo = list(subido.keys())[0]
print("Archivo cargado correctamente:", nombre_archivo)

Archivo cargado correctamente: Asaltos 1 (2).xlsx


# Buscar la hoja donde esta el folio y estado

In [5]:
df = None
excel = pd.ExcelFile(nombre_archivo)
for hoja in excel.sheet_names:
    df_temp = pd.read_excel(nombre_archivo, sheet_name=hoja, header=None)
    for i, row in df_temp.iterrows():
        valores = [str(v).strip().upper() for v in row.values]
        if 'FOLIO' in valores and 'ESTADO' in valores:
            df = pd.read_excel(nombre_archivo, sheet_name=hoja, skiprows=i)
            print(f"Datos encontrados en la hoja '{hoja}', fila {i+1}")
            break
    if df is not None:
        break

# Limpiar nombres de columnas
df.columns = [str(col).strip().upper() for col in df.columns]

Datos encontrados en la hoja 'Sheet1', fila 1


#Filtrar ciudad de México y Estado de México

In [6]:
df['ESTADO'] = df['ESTADO'].astype(str).str.strip().str.upper()
df_filtrado = df[df['ESTADO'].isin(['CIUDAD DE MEXICO', 'ESTADO DE MEXICO', 'CIUDAD DE MÉXICO', 'ESTADO DE MÉXICO'])].copy()

df_filtrado['LATITUD'] = pd.to_numeric(df_filtrado['LATITUD'], errors='coerce')
df_filtrado['LONGITUD'] = pd.to_numeric(df_filtrado['LONGITUD'], errors='coerce')
df_filtrado = df_filtrado.dropna(subset=['LATITUD', 'LONGITUD'])
df_img = df_filtrado[(df_filtrado['LATITUD'] > 18.0) & (df_filtrado['LATITUD'] < 21.0)].head(1000)

print(f"Se encontraron {len(df_img)} registros válidos. Descargando imágenes...")

Se encontraron 1000 registros válidos. Descargando imágenes...


# Descargar los mapas estaticos con OpenStreetMap

In [7]:
# Asegurar que las dependencias estén instaladas automáticamente en tiempo de ejecución
#try:
 #   from staticmap import StaticMap, CircleMarker
#except ImportError:
 #   print("Instalando dependencias necesarias (staticmap)...")
  #  subprocess.check_call([sys.executable, "-m", "pip", "install", "staticmap", "pillow"])
   # from staticmap import StaticMap, CircleMarker

os.makedirs("mapas", exist_ok=True)
descargadas = 0

# Crear mapas de 600x400 píxeles
for idx, row in df_img.iterrows():
    lat, lon = row['LATITUD'], row['LONGITUD']
    folio = str(row['FOLIO']).strip()
    riesgo = str(row.get('TIPO DE RIESGO', 'MEDIO')).strip().upper()

    try:
        # Inicializar mapa estático libre de OpenStreetMap sin parámetros inválidos
        mapa = StaticMap(600, 400)

        # Agregar un marcador rojo
        marcador = CircleMarker((lon, lat), '#FF0000', 12)
        mapa.add_marker(marcador)

        # Renderizar y guardar imagen
        imagen_mapa = mapa.render(zoom=17)
        ruta_destino = f"mapas/{riesgo}_{folio}_{idx}.jpg"
        imagen_mapa.save(ruta_destino)
        descargadas += 1
    except Exception as e:
        print(f"No se pudo generar el mapa para el folio {folio}: {e}")

print(f"\n¡Completado! Se generaron {descargadas} imágenes de mapas.")


¡Completado! Se generaron 1000 imágenes de mapas.


#Creación de la carpeta para cada clase

In [8]:
carpeta_origen = "mapas"
carpeta_dataset = "dataset_riesgo"

# Crear subcarpetas para cada clase
clases = ['BAJO', 'MEDIO', 'ALTO']
for clase in clases:
    os.makedirs(os.path.join(carpeta_dataset, clase), exist_ok=True)

#Mover cada imagen a su carpeta conforme el nombre de clasificación

In [9]:
imagenes = glob.glob(f"{carpeta_origen}/*.jpg")
conteo = {'BAJO': 0, 'MEDIO': 0, 'ALTO': 0}

for ruta_img in imagenes:
    nombre_archivo = os.path.basename(ruta_img)
    for clase in clases:
        if nombre_archivo.startswith(clase):
            destino = os.path.join(carpeta_dataset, clase, nombre_archivo)
            shutil.copy(ruta_img, destino)
            conteo[clase] += 1
            break

print("Distribución de imágenes por carpeta:", conteo)

Distribución de imágenes por carpeta: {'BAJO': 393, 'MEDIO': 1012, 'ALTO': 124}


# Entrenamiento y Validación

In [10]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 8

train_ds = tf.keras.utils.image_dataset_from_directory(
    carpeta_dataset,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    carpeta_dataset,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

nombres_clases = train_ds.class_names
print("Clases detectadas:", nombres_clases)

Found 1529 files belonging to 3 classes.
Using 1224 files for training.
Found 1529 files belonging to 3 classes.
Using 305 files for validation.
Clases detectadas: ['ALTO', 'BAJO', 'MEDIO']


Cargar modelo base preentrenado (MobileNetV2)

In [11]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Congelar sus pesos para no dañar lo que ya sabe

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


# Construcción de la arquitectura del clasificador

In [12]:
model = models.Sequential([
    layers.Rescaling(1./127.5, offset=-1), # Normalizar píxeles para MobileNetV2
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),                   # Evita el sobreajuste (memorización)
    layers.Dense(len(nombres_clases), activation='softmax')
])

# Compilación del modelo

In [13]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

#Entrenar durante 15 épocas

In [14]:
historial = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

Epoch 1/15
153/153 ━━━━━━━━━━━━━━━━━━━━ 69s 424ms/step - accuracy: 0.6103 - loss: 0.9386 - val_accuracy: 0.6525 - val_loss: 1.0172
Epoch 2/15
153/153 ━━━━━━━━━━━━━━━━━━━━ 70s 457ms/step - accuracy: 0.6405 - loss: 0.8557 - val_accuracy: 0.6525 - val_loss: 1.0351
Epoch 3/15
153/153 ━━━━━━━━━━━━━━━━━━━━ 71s 462ms/step - accuracy: 0.6650 - loss: 0.7973 - val_accuracy: 0.6459 - val_loss: 0.8863
Epoch 4/15
153/153 ━━━━━━━━━━━━━━━━━━━━ 74s 412ms/step - accuracy: 0.6928 - loss: 0.7319 - val_accuracy: 0.6459 - val_loss: 0.8542
Epoch 5/15
153/153 ━━━━━━━━━━━━━━━━━━━━ 61s 401ms/step - accuracy: 0.6912 - loss: 0.7190 - val_accuracy: 0.6492 - val_loss: 0.9073
Epoch 6/15
153/153 ━━━━━━━━━━━━━━━━━━━━ 72s 471ms/step - accuracy: 0.6977 - loss: 0.7085 - val_accuracy: 0.6328 - val_loss: 0.8713
Epoch 7/15
153/153 ━━━━━━━━━━━━━━━━━━━━ 68s 447ms/step - accuracy: 0.7181 - loss: 0.6752 - val_accuracy: 0.6164 - val_loss: 0.8423
Epoch 8/15
153/153 ━━━━━━━━━━━━━━━━━━━━ 72s 379ms/step - accuracy: 0.7206 - loss: 0

#Generación del modelo entrenado

In [21]:
model.save("modelo_riesgo.keras")
print("¡Modelo entrenado y guardado exitosamente!")

¡Modelo entrenado y guardado exitosamente!


# Implementar preprocesamiento y postprocesamiento.

## Capa de limpieza y aumento de datos en entrenamiento

In [22]:
preprocesamiento_entrenamiento = tf.keras.Sequential([
    # Recortar un 10% los bordes para eliminar marcas de agua de OpenStreetMap
    layers.CenterCrop(height=360, width=540),
    # Redimensionar al tamaño exacto que pide MobileNetV2
    layers.Resizing(224, 224),
    # Volteos aleatorios para simular distintas orientaciones de calles
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.1),
    # Normalización matemática (-1 a 1) exigida por MobileNetV2
    layers.Rescaling(1./127.5, offset=-1)
])

## Capa de preprocesamiento estricto (para imágenes nuevas en producción)

In [23]:
preprocesamiento_prediccion = tf.keras.Sequential([
    layers.CenterCrop(height=360, width=540),
    layers.Resizing(224, 224),
    layers.Rescaling(1./127.5, offset=-1)
])

# Post procesamiento

In [25]:
# Cargar el modelo previamente guardado
modelo_cargado = tf.keras.models.load_model("modelo_riesgo.keras")
clases = ['ALTO', 'BAJO', 'MEDIO'] # ¡OJO! Usa el orden exacto que imprimió train_ds.class_names

def evaluar_nueva_ubicacion(latitud, longitud, umbral_confianza=0.65):
    # --- PASO A: GENERAR Y PREPROCESAR LA IMAGEN ---
    mapa = StaticMap(600, 400)
    mapa.add_marker(CircleMarker((longitud, latitud), '#FF0000', 12))
    img_pil = mapa.render(zoom=17)

    # Convertir imagen PIL a arreglo numérico de TensorFlow y agregar dimensión de lote (batch)
    img_array = tf.keras.utils.img_to_array(img_pil)
    img_batch = tf.expand_dims(img_array, 0)

    # Aplicar el preprocesamiento limpio (recorte, redimensión y normalización)
    img_lista = preprocesamiento_prediccion(img_batch)

    # --- PASO B: PREDICCIÓN DEL MODELO ---
    predicciones = modelo_cargado.predict(img_lista, verbose=0)[0]

    # --- PASO C: POSTPROCESAMIENTO ---
    indice_ganador = np.argmax(predicciones)
    clase_predicha = clases[indice_ganador]
    confianza = float(predicciones[indice_ganador])

    # Regla de negocio: Si la confianza es baja, pedir revisión humana
    if confianza < umbral_confianza:
        estado_final = f"REVISIÓN MANUAL (Posible {clase_predicha})"
        accion_sugerida = "El modelo no está suficientemente seguro. Validar ruta con analista."
    else:
        estado_final = clase_predicha
        if clase_predicha == 'ALTO':
            accion_sugerida = "ALERTA: Asignar custodia o cambiar horario de entrega."
        elif clase_predicha == 'MEDIO':
            accion_sugerida = "PRECAUCIÓN: Mantener monitoreo GPS activo."
        else:
            accion_sugerida = "RUTA SEGURA: Operación normal."

    # Estructurar salida limpia (ideal para exportar a Excel o API)
    resultado = {
        "coordenadas": (latitud, longitud),
        "riesgo_detectado": estado_final,
        "confianza_porcentaje": round(confianza * 100, 2),
        "desglose_probabilidades": {clases[i]: round(float(predicciones[i]) * 100, 2) for i in range(len(clases))},
        "protocolo": accion_sugerida
    }
    return resultado

# --- EJEMPLO DE USO CON UNA COORDENADA DE CDMX ---
prueba = evaluar_nueva_ubicacion(19.4326, -99.1332)
for clave, valor in prueba.items():
    print(f"{clave}: {valor}")

coordenadas: (19.4326, -99.1332)
riesgo_detectado: REVISIÓN MANUAL (Posible ALTO)
confianza_porcentaje: 57.8
desglose_probabilidades: {'ALTO': 57.8, 'BAJO': 0.51, 'MEDIO': 41.69}
protocolo: El modelo no está suficientemente seguro. Validar ruta con analista.


# Inferencia por Lote

In [26]:


# 1. Simular un archivo Excel con rutas futuras (o sube el tuyo con files.upload())
datos_rutas = pd.DataFrame({
    'ID_RUTA': ['R-101', 'R-102', 'R-103', 'R-104'],
    'CLIENTE': ['Tienda Centro', 'Abarrotes Norte', 'Super Sur', 'MiniSuper Oriente'],
    'LATITUD': [19.4326, 19.5045, 19.3574, 19.4012],
    'LONGITUD': [-99.1332, -99.1469, -99.1620, -99.0543]
})

clases = ['ALTO', 'BAJO', 'MEDIO'] # Usa el orden exacto de tu entrenamiento

def inferencia_por_lote(df_rutas, umbral_confianza=0.65):
    print(f"Generando mapas para {len(df_rutas)} paradas...")
    lote_imagenes = []

    # Paso A: Descargar y preprocesar todas las imágenes del lote
    for _, row in df_rutas.iterrows():
        mapa = StaticMap(600, 400)
        mapa.add_marker(CircleMarker((row['LONGITUD'], row['LATITUD']), '#FF0000', 12))
        img_pil = mapa.render(zoom=17)

        img_array = tf.keras.utils.img_to_array(img_pil)
        lote_imagenes.append(img_array)

    # Convertir la lista en un solo tensor de 4 dimensiones: (Cantidad, Alto, Ancho, Canales)
    tensor_lote = tf.convert_to_tensor(np.array(lote_imagenes))
    tensor_preprocesado = preprocesamiento_prediccion(tensor_lote)

    # Paso B: Ejecutar predicción masiva en una sola llamada al modelo
    print("Ejecutando predicción por lote...")
    predicciones = modelo_cargado.predict(tensor_preprocesado, batch_size=16, verbose=0)

    # Paso C: Postprocesar resultados y agregarlos al DataFrame
    riesgos_finales = []
    confianzas = []

    for pred in predicciones:
        idx = np.argmax(pred)
        conf = float(pred[idx])
        etiqueta = clases[idx] if conf >= umbral_confianza else f"REVISAR ({clases[idx]})"

        riesgos_finales.append(etiqueta)
        confianzas.append(round(conf * 100, 2))

    df_rutas['RIESGO_PREDICHO'] = riesgos_finales
    df_rutas['CONFIANZA_%'] = confianzas
    return df_rutas

# Ejecutar el proceso y guardar el resultado
df_resultado = inferencia_por_lote(datos_rutas)
print("\nResultados del lote:")
print(df_resultado)

df_resultado.to_excel("Rutas_Evaluadas_Seguridad.xlsx", index=False)
# files.download("Rutas_Evaluadas_Seguridad.xlsx") # Descomenta esta línea para descargarlo a tu PC


Generando mapas para 4 paradas...
Ejecutando predicción por lote...

Resultados del lote:
  ID_RUTA            CLIENTE  LATITUD  LONGITUD  RIESGO_PREDICHO  CONFIANZA_%
0   R-101      Tienda Centro  19.4326  -99.1332   REVISAR (ALTO)        57.80
1   R-102    Abarrotes Norte  19.5045  -99.1469  REVISAR (MEDIO)        56.45
2   R-103          Super Sur  19.3574  -99.1620   REVISAR (ALTO)        50.05
3   R-104  MiniSuper Oriente  19.4012  -99.0543   REVISAR (ALTO)        50.94


# Inferencia en Tiempo Real (Simulación de Telemetría GPS)

In [27]:
import time

# Simulación de una ruta en movimiento (coordenadas GPS enviadas cada 3 segundos)
telemetria_camion = [
    {"unidad": "BIMBO-402", "hora": "08:00:00", "lat": 19.4326, "lon": -99.1332},
    {"unidad": "BIMBO-402", "hora": "08:05:00", "lat": 19.4410, "lon": -99.1420},
    {"unidad": "BIMBO-402", "hora": "08:10:00", "lat": 19.4520, "lon": -99.1510},
    {"unidad": "BIMBO-402", "hora": "08:15:00", "lat": 19.4600, "lon": -99.1600}
]

def monitorear_en_tiempo_real(flujo_gps):
    print("Iniciando monitoreo GPS en tiempo real...\n" + "-"*50)

    for evento in flujo_gps:
        inicio_tiempo = time.time()

        # 1. Capturar entorno actual del vehículo
        mapa = StaticMap(600, 400)
        mapa.add_marker(CircleMarker((evento['lon'], evento['lat']), '#FF0000', 12))
        img_pil = mapa.render(zoom=17)

        # 2. Preprocesar unidad individual
        img_array = tf.keras.utils.img_to_array(img_pil)
        img_batch = tf.expand_dims(img_array, 0)
        img_lista = preprocesamiento_prediccion(img_batch)

        # 3. Inferencia instantánea
        pred = modelo_cargado.predict(img_lista, verbose=0)[0]
        idx = np.argmax(pred)
        riesgo = clases[idx]
        confianza = round(float(pred[idx]) * 100, 1)

        # Calcular latencia (tiempo de respuesta)
        latencia_ms = round((time.time() - inicio_tiempo) * 1000)

        # 4. Mostrar estado en consola (Simulación de Dashboard)
        alerta = "🚨 [ALERTA DE SEGURIDAD]" if riesgo == "ALTO" else "✅ [OK]"
        print(f"[{evento['hora']}] Unidad: {evento['unidad']} | Riesgo: {riesgo} ({confianza}%) | Tiempo: {latencia_ms}ms {alerta}")

        # Pausa de 2 segundos para simular la espera del siguiente paquete GPS
        time.sleep(2)

# Ejecutar la simulación en vivo
monitorear_en_tiempo_real(telemetria_camion)


Iniciando monitoreo GPS en tiempo real...
--------------------------------------------------
[08:00:00] Unidad: BIMBO-402 | Riesgo: ALTO (57.8%) | Tiempo: 407ms 🚨 [ALERTA DE SEGURIDAD]
[08:05:00] Unidad: BIMBO-402 | Riesgo: ALTO (50.4%) | Tiempo: 382ms 🚨 [ALERTA DE SEGURIDAD]
[08:10:00] Unidad: BIMBO-402 | Riesgo: MEDIO (51.6%) | Tiempo: 499ms ✅ [OK]
[08:15:00] Unidad: BIMBO-402 | Riesgo: ALTO (50.4%) | Tiempo: 660ms 🚨 [ALERTA DE SEGURIDAD]
